# Task 3 — Outlier Detection

**Default method: IQR**  
**Optional method: PyOD KNN** (`method="knn"`)

### IQR formula
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 − Q1
- Lower = Q1 − 1.5×IQR
- Upper = Q3 + 1.5×IQR
- Values outside bounds = outliers (NaNs ignored)

### Why IQR is default
In testing: catch rate 4/4, false positives 0. Faster + more explainable than ML for Phase 1.

### Why keep KNN
Useful comparison / future work, but slower on large files — use only on small samples in demos.

### Depends on Task 1
Runs on columns after header load. Headerless scrap sheets (`unnamed_*`) still work if values are numeric.  
100% null columns (Task 2 completeness) produce **no outliers** (`reason=no_numeric`) — expected.

**Output:** per-column `CheckResult` objects showing IQR/KNN bounds, flagged row indices/values, and (new, see below) which columns were skipped because they aren't real measurements.


In [1]:
from dataclasses import dataclass, field
from typing import Any

import pandas as pd


@dataclass
class CheckResult:
    check_name: str
    status: str
    column: str | None
    issues_found: int
    details: dict[str, Any] = field(default_factory=dict)
    dimension: str = ""


def _to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def detect_outliers_iqr(series: pd.Series, multiplier: float = 1.5) -> CheckResult:
    col = str(series.name) if series.name is not None else None
    try:
        numeric = _to_numeric(series)
        valid = numeric.dropna()
        if len(valid) == 0:
            return CheckResult("outliers", "passed", col, 0, {"reason": "no_numeric", "method": "iqr"}, "validity")
        if len(valid) < 4:
            return CheckResult(
                "outliers", "passed", col, 0,
                {"reason": "insufficient_numeric_values", "method": "iqr", "numeric_count": len(valid)},
                "validity",
            )

        q1 = float(valid.quantile(0.25))
        q3 = float(valid.quantile(0.75))
        iqr = q3 - q1
        lower = q1 - multiplier * iqr
        upper = q3 + multiplier * iqr

        mask = ((numeric < lower) | (numeric > upper)).fillna(False)
        idx = series.index[mask].tolist()
        vals = [float(v) for v in numeric[mask].tolist()]
        n = len(idx)
        pct = round((n / len(valid)) * 100, 4)

        return CheckResult(
            "outliers",
            "passed" if n == 0 else "failed",
            col,
            n,
            {
                "method": "iqr",
                "q1": q1,
                "q3": q3,
                "iqr": iqr,
                "lower_bound": lower,
                "upper_bound": upper,
                "outlier_count": n,
                "outlier_pct": pct,
                "row_indices": idx[:100],
                "sample_values": vals[:20],
                "constant_column": iqr == 0.0,
            },
            "validity",
        )
    except Exception as e:
        return CheckResult("outliers", "error", col, 0, {"error": str(e), "method": "iqr"}, "validity")


def detect_outliers_knn(series: pd.Series, n_neighbors: int = 5, contamination: float = 0.05) -> CheckResult:
    """Optional comparator. Needs pyod installed."""
    col = str(series.name) if series.name is not None else None
    try:
        from pyod.models.knn import KNN
    except ImportError:
        return CheckResult(
            "outliers", "error", col, 0,
            {"error": "pyod not installed (optional). Default remains IQR.", "method": "knn"},
            "validity",
        )

    try:
        numeric = _to_numeric(series)
        valid = numeric.dropna()
        if len(valid) <= n_neighbors:
            return CheckResult(
                "outliers", "passed", col, 0,
                {"reason": "insufficient_for_knn", "method": "knn"},
                "validity",
            )

        cont = min(max(contamination, 1.0 / len(valid)), 0.5)
        X = valid.to_numpy(dtype=float).reshape(-1, 1)
        model = KNN(n_neighbors=n_neighbors, contamination=cont)
        model.fit(X)

        idx = [valid.index[i] for i, lab in enumerate(model.labels_) if int(lab) == 1]
        n = len(idx)
        return CheckResult(
            "outliers",
            "passed" if n == 0 else "failed",
            col,
            n,
            {
                "method": "knn",
                "outlier_count": n,
                "outlier_pct": round((n / len(valid)) * 100, 4),
                "n_neighbors": n_neighbors,
                "contamination": cont,
                "row_indices": idx[:100],
                "sample_values": [float(valid.loc[i]) for i in idx[:20]],
            },
            "validity",
        )
    except Exception as e:
        return CheckResult("outliers", "error", col, 0, {"error": str(e), "method": "knn"}, "validity")


def detect_outliers(series: pd.Series, method: str = "iqr") -> CheckResult:
    method = (method or "iqr").lower()
    if method == "iqr":
        return detect_outliers_iqr(series)
    if method == "knn":
        return detect_outliers_knn(series)
    return CheckResult("outliers", "error", None, 0, {"error": f"unknown method {method}"}, "validity")


print("Task 3 outlier functions ready (default=iqr)")

Task 3 outlier functions ready (default=iqr)


## IQR test (clear spike)

In [2]:
s = pd.Series([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 1000], name="Amount")
r = detect_outliers(s, method="iqr")

print("status:", r.status)
print("outliers:", r.issues_found)
print("Q1 / Q3 / IQR:", r.details.get("q1"), r.details.get("q3"), r.details.get("iqr"))
print("bounds:", r.details.get("lower_bound"), "->", r.details.get("upper_bound"))
print("pct:", r.details.get("outlier_pct"))
print("indices:", r.details.get("row_indices"))
print("values:", r.details.get("sample_values"))

status: failed
outliers: 1
Q1 / Q3 / IQR: 3.5 8.5 5.0
bounds: -4.0 -> 16.0
pct: 9.0909
indices: [10]
values: [1000.0]


## Edge cases

In [3]:
cases = {
    "constant": pd.Series([5, 5, 5, 5, 5, 5], name="Const"),
    "all_nan": pd.Series([None, None, None], name="Empty"),
    "tiny": pd.Series([1, 2], name="Tiny"),
    "negative_spike": pd.Series([-10, -1, 0, 1, 2, 3, 4, 5, -999], name="Neg"),
}
for name, series in cases.items():
    res = detect_outliers(series)
    print(f"{name:15} {res.status:7} issues={res.issues_found} details={ {k: res.details.get(k) for k in ['reason','method','outlier_pct','sample_values'] if k in res.details or res.details.get(k) is not None} }")

constant        passed  issues=0 details={'method': 'iqr', 'outlier_pct': 0.0, 'sample_values': []}
all_nan         passed  issues=0 details={'reason': 'no_numeric', 'method': 'iqr'}
tiny            passed  issues=0 details={'reason': 'insufficient_numeric_values', 'method': 'iqr'}
negative_spike  failed  issues=2 details={'method': 'iqr', 'outlier_pct': 22.2222, 'sample_values': [-10.0, -999.0]}


## Optional KNN (small sample only)
Do not run this on full Booked Orders — it can hang.

In [4]:
s2 = pd.Series([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 1000, 1100], name="Amount")
k = detect_outliers(s2, method="knn")
print("status:", k.status)
print("method:", k.details.get("method"))
print("issues:", k.issues_found)
print("error:", k.details.get("error"))
print("indices:", k.details.get("row_indices"))

status: failed
method: knn
issues: 1
error: None
indices: [11]


## Quick comparison

| | IQR (default) | PyOD KNN |
|---|---|---|
| Catch rate (our test) | 4/4 | optional comparator |
| False positives | 0 | depends on settings |
| Explainability | high | lower |
| Speed on large files | fast | slow |
| Phase 1 choice | **YES** | optional only |

### Pipeline notes (from real-file review)
- Shape / missing % on transactional sheets matched hand checks — profiling + IQR sit on top of Task 1 load.
- Install `python-calamine` so Task 1 can open broken Sage X3 workbooks before Task 2/3 run.
- Analysis-only sheets (notes/mappings) can be skipped — not raw transactional data.

Same logic is packaged in `data_quality_engine/engine/checks/outliers.py`.

## Column Classification Fix (Issue 1)

**Problem this fixes:** the functions above run IQR/KNN on *any* column with numeric-looking values -- including identifier columns (invoice numbers, customer codes, phone numbers, postal codes). Flagging an invoice number as a statistical outlier is meaningless.

**Fix (in the actual pipeline, not reimplemented here):** a new module, `engine/column_classifier.py`, classifies every column as `identifier`, `measurement`, `categorical`, `date`, `pii`, or `free_text` using column-name hints + cardinality ratio (and reuses the existing PII detector for the `pii` role). `engine/checks/outliers.py -> detect_outliers_frame()` calls this classifier first and skips any non-`measurement` column, returning a passed result with `reason: "skipped_non_measurement_column"`.

The cells below import the **real package functions** (not a notebook copy) so this demo reflects exactly what `main.py` runs.

In [5]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from data_quality_engine.engine.column_classifier import classify_columns
from data_quality_engine.engine.checks.outliers import detect_outliers_frame

print("Column classifier + frame-level outlier detection imported from the package")

Column classifier + frame-level outlier detection imported from the package


### Fabricated sheet: identifier + PII + real measurement columns

`invoice_no` and `phone` both contain numeric-looking values with high cardinality, so before this fix they would have been scanned by IQR. `Amount` is a genuine measurement with one real spike.

In [6]:
demo_df = pd.DataFrame(
    {
        "invoice_no": [100000 + i for i in range(15)],
        "phone": ["0300-1234567"] * 15,
        "Amount": [10, 11, 12, 9, 10, 11, 13, 10, 9, 12, 1000, 11, 10, 9, 12],
    }
)

roles = classify_columns(demo_df)
print("Classified roles:", roles)

Classified roles: {'invoice_no': 'identifier', 'phone': 'pii', 'Amount': 'measurement'}


In [7]:
frame_results = detect_outliers_frame(demo_df)
for r in frame_results:
    reason = r.details.get("reason")
    role = r.details.get("classified_role")
    print(f"{r.column:15} status={r.status:7} issues={r.issues_found}"
          + (f"  reason={reason} (role={role})" if reason else ""))

invoice_no      status=passed  issues=0  reason=skipped_non_measurement_column (role=identifier)
Amount          status=failed  issues=1


**Expected:** `invoice_no` is skipped with `reason=skipped_non_measurement_column` (classified `identifier`). `Amount` is scanned normally and flags the `1000` spike. `phone` doesn't even appear in the results -- it has no numeric-parseable values at all, so it's filtered out by the pre-existing "no numeric values" check before the classifier is consulted (same as any text-only column, unchanged by this fix). Try changing `phone` to numeric-looking digits-only values (e.g. `"03001234567"` stays text, but something like `300012345`) to see it get skipped *by the classifier* instead, with `role=pii`.